# 🚯 Lecture 11 Lab: Logistic regression and spam detection

<img src="https://github.com/joshuagrossman/mse125-labs-public/blob/main/hw5/img/spam-email.png?raw=1" alt= “spam-email” width="500" />

## ✅ Setup and data import
In this lab, we will work with a [classic dataset](https://archive.ics.uci.edu/dataset/94/spambase) of 4,601 emails classified as spam or not spam.

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display floats with 3 digits, avoid scientific notation
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Set seaborn theme for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 14

# Read in the spam dataset
spam = pd.read_csv('https://jdgrossman.com/assets/spam.csv')

# Peek at 10 random rows
spam.sample(10)


,make,address,all,3d,our,over,remove,internet,order,mail,...,char_semicolon,char_left_paren,char_left_bracket,char_exclamation,char_dollar,char_pound,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
586,0.000,0.000,0.360,0.000,0.360,0.000,0.000,0.000,0.360,0.360,...,0.000,0.125,0.000,0.125,0.000,0.000,1.287,5,94,1
4186,0.380,0.000,1.160,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,2.000,51,114,0
1965,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1,4,0
3454,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.030,0.060,0.000,0.000,0.000,0.000,2.444,76,198,0
4590,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.185,0.000,0.000,0.000,0.092,2.468,11,79,0
1196,0.000,0.360,0.720,0.000,1.440,0.000,0.360,0.000,0.000,1.440,...,0.000,0.000,0.000,0.000,0.000,2.517,6.685,60,234,1
3172,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,2.428,5,17,0
613,0.110,0.220,0.110,0.000,0.450,0.450,0.000,0.110,1.020,1.590,...,0.018,0.170,0.000,0.265,0.132,0.000,4.215,144,666,1
2384,0.000,0.000,0.000,0.000,0.540,0.000,0.000,0.000,0.000,1.630,...,0.000,0.090,0.090,0.000,0.000,0.000,1.969,16,65,0
911,0.200,0.400,0.400,0.000,0.000,0.400,0.000,0.200,1.430,0.610,...,0.029,0.059,0.447,0.298,0.149,0.029,11.960,376,909,1


## ♨️ Warm up

How many emails are in the database?

What fraction of the emails in the database are spam?

Which email contains the highest percentage of words matching "money"? What percentage of words in that email match "money"?

In [3]:
# Your code here!

# Total number of emails
num_emails = spam.shape[0]
print("Number of emails:", num_emails)

# Fraction of emails that are spam
spam_fraction = spam['is_spam'].mean()
print("Fraction of emails that are spam:", spam_fraction)

# Email with the highest percentage of words matching "money"
# The 'money' column already contains the fraction of words in the email that are 'money'
max_money_idx = spam['money'].idxmax()
max_money_fraction = spam.loc[max_money_idx, 'money']  # fraction of words

print("Index of email with highest % of 'money':", max_money_idx)
print("Percentage of words that are 'money':", max_money_fraction * 100, "%")


Number of emails: 4601
Fraction of emails that are spam: 0.39404477287546186
Index of email with highest % of 'money': 545
Percentage of words that are 'money': 1250.0 %


## 🎲 Linear probability models (LPMs)

Fit a linear regression model to the spam data with the `lm` function.

Use the following covariates to predict the likelihood that an email is spam:
- `char_dollar`
- `credit`
- `money`
- `re`

How would you interpret the model coefficients for the intercept and for `char_dollar`?

- Note: `char_dollar` represents the percentage of characters in the email that match `$`.

In [4]:
# Your code here!
import statsmodels.api as sm

# Define predictors
X = spam[['char_dollar', 'credit', 'money', 're']]

# Add a constant term for the intercept
X = sm.add_constant(X)

# Define the response variable
y = spam['is_spam']

# Fit linear regression (LPM)
lpm_model = sm.OLS(y, X).fit()

# Print summary
print(lpm_model.summary())



                            OLS Regression Results                            
Dep. Variable:                is_spam   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     250.3
Date:                Tue, 18 Nov 2025   Prob (F-statistic):          8.51e-195
Time:                        05:14:18   Log-Likelihood:                -2780.3
No. Observations:                4601   AIC:                             5571.
Df Residuals:                    4596   BIC:                             5603.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.3346      0.007     45.696      

Using your linear probability model and the `predict` function, predict the in-sample probability that each email is spam.

What is the smallest predicted probability? The largest? Do you notice any issues with these predictions?

In [5]:
# Your code here!
# Predict in-sample probabilities using the LPM
pred_probs = lpm_model.predict(X)

# Add predicted probabilities to the dataset (optional)
spam['pred_prob'] = pred_probs

# Smallest and largest predicted probabilities
min_prob = pred_probs.min()
max_prob = pred_probs.max()

print("Smallest predicted probability:", min_prob)
print("Largest predicted probability:", max_prob)



Smallest predicted probability: -0.8125403869560923
Largest predicted probability: 3.8494089964108236


## 🎰 Odds functions

Write two functions:
- A function to convert probabilities to odds.
- A function to convert odds to probabilities

Test your functions by making sure that 2:1 odds returns a 2/3 probability, and vice versa.

Finally, suppose my probability of winning is 60%. If I double my odds of winning, what is my new probability of winning?

In [6]:
# Your code here!

# Function to convert probability to odds
def prob_to_odds(p):
    return p / (1 - p)

# Function to convert odds to probability
def odds_to_prob(o):
    return o / (1 + o)


# Test 2:1 odds → probability
test_odds = 2
prob_from_odds = odds_to_prob(test_odds)
print("Probability from 2:1 odds:", prob_from_odds)  # Expected 2/3 ≈ 0.6667

# Test probability 2/3 → odds
test_prob = 2/3
odds_from_prob = prob_to_odds(test_prob)
print("Odds from 2/3 probability:", odds_from_prob)  # Expected 2


# Original probability
p = 0.6

# Convert to odds
o = prob_to_odds(p)

# Double the odds
new_o = 2 * o

# Convert back to probability
new_p = odds_to_prob(new_o)
print("New probability if odds are doubled:", new_p)


Probability from 2:1 odds: 0.6666666666666666
Odds from 2/3 probability: 1.9999999999999998
New probability if odds are doubled: 0.75


## 🪙 Fitting a logistic regression model

We can fit a logistic regression model with the same covariates as above with the following code:

In [ ]:
model = glm(is_spam ~ 1 + char_dollar + credit + money + re, family='binomial', data=spam)

summary(model)

Interpret the intercept and `money` coefficients for the logistic regression model three different ways:
1. On the log odds scale
2. On the odds scale (by exponentiating the coefficients)
3. On the probability scale (using either the odds functions you wrote, or the divide by 4 trick).

Tip: Use the `coef` function to extract coefficients from the model.

In [7]:
# Your code here!
import statsmodels.api as sm

# Define predictors (same as before)
X = spam[['char_dollar', 'credit', 'money', 're']]
X = sm.add_constant(X)

# Response variable
y = spam['is_spam']

# Fit logistic regression
logit_model = sm.Logit(y, X).fit()

# Show summary
print(logit_model.summary())


coeffs = logit_model.params
print(coeffs)


odds_coeffs = np.exp(coeffs)
print(odds_coeffs)


# Baseline odds
baseline_odds = np.exp(coeffs['const'])
baseline_prob = odds_to_prob(baseline_odds)

# Odds after 1-unit increase in money
new_odds = baseline_odds * np.exp(coeffs['money'])
new_prob = odds_to_prob(new_odds)



Optimization terminated successfully.
         Current function value: 0.481178
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                is_spam   No. Observations:                 4601
Model:                          Logit   Df Residuals:                     4596
Method:                           MLE   Df Model:                            4
Date:                Tue, 18 Nov 2025   Pseudo R-squ.:                  0.2824
Time:                        05:16:51   Log-Likelihood:                -2213.9
converged:                       True   LL-Null:                       -3085.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.0666      0.043    -24.680      0.000      -1.151      -0.982
char_dollar    11.8176    